In [44]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.compose import ColumnTransformer

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [23]:
train.head()

,Patient_ID,Age,Gender,Symptoms,Symptom_Count,Disease
0,P0786,22,Male,"fever, cough, sneezing, runny nose",4,Common Cold
1,P0097,86,Male,"increased thirst, fatigue",2,Diabetes
2,P0257,13,Other,"headache, sensitivity to light, nausea",3,Migraine
3,P0763,38,Other,"fever, fatigue",2,Pneumonia
4,P0895,25,Other,"blurred vision, increased thirst",2,Diabetes


In [50]:
# Tokenizer
le = LabelEncoder()

def splitString(text):
    if pd.isna(text):
        return []
    return [item.strip() for item in str(text).split(',')]

x_train = train.drop(columns=['Patient_ID', 'Disease'])
y_train = le.fit_transform(train['Disease'])
x_test = test.drop(columns=['Patient_ID'])

preprocessor = ColumnTransformer(
    transformers=[
        ('oneHot', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Gender']),
        ('list', CountVectorizer(tokenizer=splitString, token_pattern=None, binary=True), 'Symptoms')
    ]
)

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.05, 0.1]
}

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', XGBClassifier(random_state=42))
])

gs = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5
)

gs.fit(x_train, y_train)
model = gs.best_estimator_
predictions = model.predict(x_test)

In [51]:
rows = []
predictions = le.inverse_transform(predictions)
for id, pred in zip(test['Patient_ID'], predictions):
    rows.append({
        'Patient_ID': id,
        'Disease': pred
    })
sub = pd.DataFrame(rows)
sub.to_csv('submission.csv', index=False)